In [67]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [68]:
import pandas as pd
import mne
import pathlib
from scripts.s00_mne_info_builder import build_info, raw_from_data_and_metadata
from scripts.s01_preprocess import preprocess
from scripts.s02_epoch import build_binary_epochs
from scripts.s03_features_binary import evaluate

In [69]:
SUBJECT_ID = 2
INPUT_FOLDER = f'../LSLService/data/subject_{SUBJECT_ID}/'
EEG_STREAM_NAME = 'WS-default'
TIMESTAMP = '20260427_141637'

EEG_PATH = INPUT_FOLDER + f'eeg_{EEG_STREAM_NAME}_{TIMESTAMP}.csv'
EEG_METADATA_PATH = INPUT_FOLDER + f'eeg_{EEG_STREAM_NAME}_{TIMESTAMP}_metadata.txt'
MARKERS_PATH = INPUT_FOLDER + f'markers_{TIMESTAMP}.csv'

OUTPUT_PREFIX = f'data/subject_{SUBJECT_ID}_{TIMESTAMP}'

In [70]:
eeg = pd.read_csv(EEG_PATH)
eeg

,timestamp,P3,C3,F3,Fz,F4,C4,P4,Cz,Pz,...,O2,X3,X2,F7,F8,X1,A2,T6,T4,TRG
0,1426.735628,486.978210,-374.079559,-241.356583,-156.299988,-171.092163,-361.004761,-279.549957,-361.053345,-120.740791,...,-1556.735474,-9.632,-5.418,1232.319214,-1341.487915,-8.127,-23.405046,-541.466919,-284.053894,0.0
1,1426.735630,493.604492,-372.271912,-237.432831,-153.887711,-160.539246,-349.560028,-273.514679,-358.641083,-118.328522,...,-1552.516602,-10.535,-5.719,1250.678101,-1333.358521,-8.127,-21.595472,-538.148560,-278.321014,0.0
2,1426.735632,497.828400,-364.426819,-231.401627,-149.062851,-160.538940,-344.734344,-267.785767,-352.006470,-112.599617,...,-1547.992065,-10.234,-5.418,1258.212280,-1329.134155,-8.127,-17.976009,-536.346008,-276.214600,0.0
3,1426.735634,497.218567,-367.134430,-236.830017,-152.982605,-163.553711,-347.149506,-269.288452,-354.113190,-117.423416,...,-1546.191406,-10.535,-5.418,1267.229004,-1328.842651,-8.127,-19.484974,-537.545471,-275.302826,0.0
4,1426.735635,508.225922,-358.255096,-229.159576,-140.471649,-151.947906,-334.035309,-255.270203,-340.396820,-103.707069,...,-1533.077271,-10.234,-5.418,1282.148926,-1326.560425,-7.826,-19.027985,-527.453491,-268.831421,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
91678,1732.320079,-23.815540,-248.677032,-46.168510,-767.027100,1028.718872,68.208702,-293.688263,-276.204620,-730.262512,...,-546.284180,-9.933,-5.117,248.201126,-192.857010,-8.127,32.451931,-24.421215,20.796612,0.0
91679,1732.320081,-23.804661,-246.277847,-49.512669,-760.093262,1029.924194,67.916603,-294.000427,-277.723969,-723.328674,...,-542.963623,-9.933,-5.418,247.312103,-196.153687,-8.127,33.660179,-26.246170,19.579931,0.0
91680,1732.349907,-22.894238,-244.172623,-48.011902,-754.967773,1030.527588,71.836945,-291.892456,-275.615845,-719.107239,...,-543.559998,-9.933,-5.418,247.020325,-188.321472,-7.826,36.676205,-25.649094,21.084499,0.0
91681,1732.349914,-19.574966,-239.358856,-46.825535,-748.338196,1034.744751,75.456223,-287.377441,-277.440582,-711.874939,...,-543.554321,-10.234,-5.719,255.456116,-186.204483,-7.826,29.142372,-27.175657,22.883064,0.0


In [71]:
eeg_no_ts = eeg.drop('timestamp', axis=1)
eeg_info, _ = build_info(EEG_METADATA_PATH)

eeg_data = raw_from_data_and_metadata(eeg_no_ts.transpose().to_numpy(), EEG_METADATA_PATH)
eeg_preprocessed = preprocess(eeg_data, pathlib.Path(f'{OUTPUT_PREFIX}.fif'), l_freq=0.1)
eeg_preprocessed

Creating RawArray with float64 data, n_channels=24, n_times=91683
    Range : 0 ... 91682 =      0.000 ...   305.607 secs
Ready.
Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 0.1 - 40 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 0.10
- Lower transition bandwidth: 0.10 Hz (-6 dB cutoff frequency: 0.05 Hz)
- Upper passband edge: 40.00 Hz
- Upper transition bandwidth: 10.00 Hz (-6 dB cutoff frequency: 45.00 Hz)
- Filter length: 9901 samples (33.003 s)

Filtering raw data in 1 contiguous segment
Setting up band-stop filter

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandstop filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lowe

C:\Users\nbaeriswyl\github\VIBE\DataAnalysis\scripts\s01_preprocess.py:22: RuntimeWarning: Using n_components=20 (resulting in n_components_=20) may lead to an unstable mixing matrix estimation because the ratio between the largest (12) and smallest (2e-30) variances is too large (> 1e6); consider setting n_components=0.999999 or an integer <= 16
  uv run python s01_preprocess.py --input data/raw/sub-01.edf \\
C:\Users\nbaeriswyl\github\VIBE\DataAnalysis\scripts\s01_preprocess.py:33: RuntimeWarning: This filename (C:\Users\nbaeriswyl\github\VIBE\DataAnalysis\data\subject_2_20260427_141637.fif) does not conform to MNE naming conventions. All raw files should end with raw.fif, raw_sss.fif, raw_tsss.fif, _meg.fif, _eeg.fif, _ieeg.fif, raw.fif.gz, raw_sss.fif.gz, raw_tsss.fif.gz, _meg.fif.gz, _eeg.fif.gz or _ieeg.fif.gz
  


<RawArray | 24 x 91683 (305.6 s), ~16.8 MiB, data loaded>

In [76]:
markers = pd.read_csv(MARKERS_PATH)
pattern = '"PinchLog"'

markers = markers[markers["value"].str.contains(pattern, regex=False, na=False)]

markers.timestamp -= eeg.timestamp.min()
# markers

epochs = build_binary_epochs(
    eeg_preprocessed, markers.timestamp, pathlib.Path(f'{OUTPUT_PREFIX}-epo.fif'),
    tmin=-1,
    tmax=0.2,
)


Not setting metadata
131 matching events found
Setting baseline interval to [-1.0, 0.0] s
Applying baseline correction (mode: mean)
0 projection items activated
Using data from preloaded Raw for 131 events and 361 original time points ...
    Rejecting  epoch based on EEG : ['P3', 'F3', 'Fp1', 'P8']
    Rejecting  epoch based on EEG : ['P3']
    Rejecting  epoch based on EEG : ['P8']
    Rejecting  epoch based on EEG : ['F3', 'Cz', 'A2']
    Rejecting  epoch based on EEG : ['F3', 'Cz']
    Rejecting  epoch based on EEG : ['F3']
    Rejecting  epoch based on EEG : ['F3']
    Rejecting  epoch based on EEG : ['P8']
    Rejecting  epoch based on EEG : ['P8']
    Rejecting  epoch based on EEG : ['P8']
    Rejecting  epoch based on EEG : ['P8']
    Rejecting  epoch based on EEG : ['Fp2', 'T7', 'P8', 'T8']
    Rejecting  epoch based on EEG : ['F3', 'Fp1', 'T7', 'P8']
    Rejecting  epoch based on EEG : ['T7']
    Rejecting  epoch based on EEG : ['T7']
15 bad epochs dropped
Built 116 epochs: 7

In [77]:
evaluate(pathlib.Path(f'{OUTPUT_PREFIX}-epo.fif'))

Reading C:\Users\nbaeriswyl\github\VIBE\DataAnalysis\data\subject_2_20260427_141637-epo.fif ...
    Found the data of interest:
        t =   -1000.00 ...     200.00 ms
        0 CTF compensation matrices available
Not setting metadata
116 matching events found
No baseline correction applied
0 projection items activated
Base rate (positive class): 0.681

  BandPower+Hjorth + LogReg
    ROC-AUC = 0.823   PR-AUC = 0.856
    Sensitivity @ FPR=1%: 0.00   @ FPR=5%: 0.13   @ FPR=10%: 0.25
    At Youden threshold 0.28: sens=0.91, spec=0.68, prec=0.86
  TangentSpace + LogReg
    ROC-AUC = 0.778   PR-AUC = 0.876
    Sensitivity @ FPR=1%: 0.05   @ FPR=5%: 0.42   @ FPR=10%: 0.49
    At Youden threshold 0.91: sens=0.66, spec=0.76, prec=0.85
